# 05 — Structured LLM calls with `FakeProvider`

`StructuredLLM` is the LLM adapter. It is provider-agnostic and:

1. Injects the response model's JSON schema into the request.
2. Validates the returned text against a Pydantic `response_model`.
3. **Retries on validation error** up to `max_retries`, appending the validator's
   message so the model can self-correct.
4. Emits `llm.call.started` → `llm.call.completed` provenance events with
   `prompt_tokens`, `completion_tokens`, `cost_usd_micro`, `attempts`,
   `budget_before`, `budget_after`.
5. **Debits the task's budget** using integer-precise micro-USD math, so the
   JSONL invariant *budget never negative* holds exactly.

This notebook uses `FakeProvider`, which is what CI uses. For local LM Studio, swap in `LMStudioProvider(base_url='http://localhost:1234/v1')` — the rest of the code is identical.

**Kernel:** standard `python3`.

In [ ]:
import json, tempfile, shutil
from pathlib import Path

from pydantic import BaseModel, Field

from agent_kernel.api import AgentKernel
from agent_kernel.llm import FakeProvider, StructuredLLM, LLMCallError
from agent_kernel.models.event import EventType

workspace = Path(tempfile.mkdtemp(prefix='ak-ex05-'))
ak = AgentKernel(workspace)
task = ak.create_task(notebook_path=str(workspace / 'host.ipynb'), kernel_name='python3')
print('task:', task.task_id)

## 1. Define the response schema

`response_model` is any `pydantic.BaseModel`. Its `model_json_schema()` is what the adapter forwards to the provider.

In [ ]:
class Sentiment(BaseModel):
    label: str = Field(description='positive | negative | neutral')
    confidence: float = Field(ge=0.0, le=1.0)

Sentiment.model_json_schema()

## 2. Single happy-path call against `FakeProvider`

`FakeProvider(script=[...])` returns each JSON string from the script in order. We pass `task_id` so the call is accounted to that task's budget.

In [ ]:
provider = FakeProvider(
    script=['{"label": "positive", "confidence": 0.92}'],
    cost_usd_micro_per_call=250,   # $0.00025 per call
)
llm = StructuredLLM(provider, agent_kernel=ak, model='fake-1')

result = llm.generate(
    messages=[{'role': 'user', 'content': 'Classify: I love this product.'}],
    response_model=Sentiment,
    task_id=task.task_id,
)
print(result)
print('type:', type(result).__name__)

## 3. Retry-on-validation-error

Script an *invalid* response first, then a valid one. The adapter will see the first response fail Pydantic validation, append the validator's error to the conversation, ask the provider again, and succeed on attempt 2. The `llm.call.completed` event records `attempts=2`.

In [ ]:
retry_provider = FakeProvider(
    script=[
        '{"label": "positive", "confidence": 1.5}',   # invalid: confidence > 1
        '{"label": "positive", "confidence": 0.7}',   # valid
    ],
    cost_usd_micro_per_call=100,
)
retry_llm = StructuredLLM(retry_provider, agent_kernel=ak, model='fake-1', max_retries=2)

out = retry_llm.generate(
    messages=[{'role': 'user', 'content': 'Classify: this is fine.'}],
    response_model=Sentiment,
    task_id=task.task_id,
)
print(out)

## 4. Look at the ledger entries

Per call, you should see one `llm.call.started`, one `llm.call.completed`, and one `budget.debited`. The completed event carries usage and the budget_before/after pair, so you can audit cost accounting offline by replaying JSONL alone.

In [ ]:
events = ak.list_events(task.task_id)
for e in events:
    if e.event_type in (EventType.llm_call_started, EventType.llm_call_completed, EventType.budget_debited):
        print(f'{e.event_type.value:25s}  {json.dumps(e.payload, sort_keys=True)[:160]}')

## 5. Out-of-retries failure

Three invalid responses in a row with `max_retries=2` (i.e. 3 attempts) will raise `LLMCallError`. The cost incurred is still debited; a `llm.call.completed` event is emitted with `status=error`.

In [ ]:
always_bad = FakeProvider(
    script=[
        '{"label": "bad", "confidence": 99}',
        '{"label": "bad", "confidence": 99}',
        '{"label": "bad", "confidence": 99}',
    ],
    cost_usd_micro_per_call=50,
)
bad_llm = StructuredLLM(always_bad, agent_kernel=ak, model='fake-1', max_retries=2)

try:
    bad_llm.generate(
        messages=[{'role': 'user', 'content': '...'}],
        response_model=Sentiment,
        task_id=task.task_id,
    )
except LLMCallError as exc:
    print('expected failure:', str(exc)[:200])

## Pointer to local LM Studio

Swap one line to talk to a real local model:

```python
from agent_kernel.llm import LMStudioProvider

provider = LMStudioProvider(base_url='http://localhost:1234/v1')
if provider.is_reachable():
    llm = StructuredLLM(provider, agent_kernel=ak, model='your-local-model')
    # … same generate() call …
```

`is_reachable()` lets you skip cleanly when LM Studio isn't running, which is exactly what the optional `tests/integration/test_m7_llm.py::test_lmstudio_*` test does.

In [ ]:
shutil.rmtree(workspace, ignore_errors=True)